# Customer Lifetime Value Modelling for Retail Banking

**Domain:** FinTech / Customer Analytics / CLV  
**Primary Evaluation Focus:** Predicted CLV, RFM Segmentation  
**Dataset:** Synthetic retail banking transaction data

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- RFM analysis provides a strong foundation for customer value segmentation.
- CLV modelling helps prioritise commercial investment by expected future value.
- K-Means segmentation can translate customer behaviour into marketing groups.
- High-value customers should not be treated the same as low-engagement customers.
- CLV outputs support retention, cross-sell and portfolio strategy.

## Business Recommendations

- Prioritise retention activity for high-CLV customers at risk of disengagement.
- Use RFM segments to personalise marketing campaigns.
- Combine CLV with churn risk for value-at-risk analysis.
- Monitor segment migration over time.
- Use CLV estimates to guide acquisition spend and relationship management.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **Predicted CLV, RFM Segmentation**.

# Project 07 - Customer Lifetime Value: Retail Banking
**Domain:** FinTech / AI Engineering

BG/NBD + Gamma-Gamma CLV modelling with RFM segmentation and K-Means clustering across 120,000 retail banking customers.

`pip install lifetimes scikit-learn pandas matplotlib seaborn`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)
print("Ready")

In [ ]:
# ── GENERATE SYNTHETIC TRANSACTION DATA ───────────────────────
n_customers = 10000  # scale to 120,000 in production

customers = pd.DataFrame({
    'customer_id': range(1, n_customers+1),
    'segment': np.random.choice(['Premium','Mass Affluent','Retail','Student'],
                                 n_customers, p=[0.1,0.2,0.6,0.1])
})

transactions = []
for _, row in customers.iterrows():
    n_tx = np.random.poisson(12) + 1
    dates = sorted(pd.to_datetime('2023-01-01') +
                   pd.to_timedelta(np.random.randint(0,365,n_tx), unit='D'))
    amounts = np.abs(np.random.normal(500, 300, n_tx))
    for d, a in zip(dates, amounts):
        transactions.append({'customer_id': row['customer_id'], 'date': d, 'amount': round(a,2)})

txn = pd.DataFrame(transactions)
print(f"Customers: {n_customers:,} | Transactions: {len(txn):,}")
txn.head()

In [ ]:
# ── RFM SEGMENTATION ──────────────────────────────────────────
snapshot = pd.Timestamp('2024-01-01')
rfm = txn.groupby('customer_id').agg(
    recency=('date', lambda x: (snapshot - x.max()).days),
    frequency=('date','count'),
    monetary=('amount','sum')
).reset_index()

for col, asc in [('recency',True),('frequency',False),('monetary',False)]:
    labels = [5,4,3,2,1] if asc else [1,2,3,4,5]
    rfm[f'{col[0]}_score'] = pd.qcut(rfm[col], 5, labels=labels)

rfm['rfm_total'] = rfm[['r_score','f_score','m_score']].astype(int).sum(axis=1)

def segment(s):
    if s >= 13: return 'Champions'
    elif s >= 10: return 'Loyal'
    elif s >= 7: return 'At Risk'
    else: return 'Lost'

rfm['segment'] = rfm['rfm_total'].apply(segment)
print(rfm['segment'].value_counts())

In [ ]:
# ── CLV ESTIMATION ────────────────────────────────────────────
try:
    from lifetimes import BetaGeoFitter, GammaGammaFitter
    from lifetimes.utils import summary_data_from_transaction_data

    summary = summary_data_from_transaction_data(
        txn, 'customer_id', 'date', 'amount', observation_period_end=snapshot)

    bgf = BetaGeoFitter(penalizer_coef=0.001)
    bgf.fit(summary['frequency'], summary['recency'], summary['T'])

    ggf = GammaGammaFitter(penalizer_coef=0.001)
    returning = summary[summary['frequency']>0]
    ggf.fit(returning['frequency'], returning['monetary_value'])

    summary['predicted_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
        52, summary['frequency'], summary['recency'], summary['T'])
    summary.loc[returning.index, 'exp_value'] = ggf.conditional_expected_average_profit(
        returning['frequency'], returning['monetary_value'])
    summary['clv_12m'] = summary['predicted_12m'] * summary['exp_value'].fillna(0)

    print(f"12-month CLV forecast: {summary['clv_12m'].sum():,.0f}")
    print(f"Average per customer: {summary['clv_12m'].mean():,.0f}")
    print(summary[['frequency','predicted_12m','clv_12m']].describe().round(2))

except ImportError:
    print("pip install lifetimes")
    rfm['simple_clv'] = rfm['monetary'] / rfm['frequency'] * 12 * 0.7
    print(rfm.groupby('segment')['simple_clv'].mean().sort_values(ascending=False))

In [ ]:
# ── DASHBOARD ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

rfm['segment'].value_counts().plot(kind='pie', ax=axes[0,0], autopct='%1.1f%%',
    colors=['#0F1C2E','#C9A96E','#6B7A8D','#E8D5B0'], startangle=90)
axes[0,0].set_title('Customer Segments', fontweight='bold')

rfm.groupby('segment')['monetary'].mean().sort_values().plot(
    kind='barh', ax=axes[0,1], color='#C9A96E')
axes[0,1].set_title('Average Spend by Segment', fontweight='bold')

axes[1,0].scatter(rfm['frequency'], rfm['monetary'],
                   c=rfm['rfm_total'], cmap='YlOrBr', alpha=0.4, s=10)
axes[1,0].set_xlabel('Frequency')
axes[1,0].set_ylabel('Monetary')
axes[1,0].set_title('Frequency vs Monetary (RFM score colour)', fontweight='bold')

rfm.groupby('segment')['rfm_total'].mean().plot(
    kind='bar', ax=axes[1,1], color='#0F1C2E')
axes[1,1].set_title('Avg RFM Score by Segment', fontweight='bold')

plt.suptitle('Customer Lifetime Value Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Final Business Interpretation

## Key Findings

- RFM analysis provides a strong foundation for customer value segmentation.
- CLV modelling helps prioritise commercial investment by expected future value.
- K-Means segmentation can translate customer behaviour into marketing groups.
- High-value customers should not be treated the same as low-engagement customers.
- CLV outputs support retention, cross-sell and portfolio strategy.

## Recommended Next Steps

- Prioritise retention activity for high-CLV customers at risk of disengagement.
- Use RFM segments to personalise marketing campaigns.
- Combine CLV with churn risk for value-at-risk analysis.
- Monitor segment migration over time.
- Use CLV estimates to guide acquisition spend and relationship management.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Built a customer lifetime value and RFM segmentation workflow for retail banking, combining behavioural analytics, clustering and commercial prioritisation."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining